**To the student**: summarize in your own words, based on the Wikipeida content quoted above, and potentially additional sources:
* What are we trying to solve in a classification problem?
* What is a classifier?

**Answer**: 
* In a classification problem, we are trying to assign labels or categories to data points/observations based on their features. The goal is to accurately predict the category of new, unseen data based on patterns learned from a training dataset.
* A classifier is a model or algorithm that is trained to recognize patterns in the data and make predictions about which category a given data point belongs to. It takes input features and outputs a predicted label, effectively serving as a decision-making tool that helps in categorizing data into predefined classes.

**To the student**: 
* What does the Boolean values (`True` and `False`) stand for in the target `y_train_5`?
* Why wasn't `X` changed as well?

**Answer**:
* The Boolean values `True` and `False` in the target `y_train_5` represent whether a given data point belongs to a specific class (e.g., "5") or not. `True` indicates that the data point is of the class being predicted, while `False` indicates that it is not.
* `X` wasn't changed because it contains the input features of the data points, which remain the same regardless of the target class being predicted. The features are used to train the classifier, while the target variable (`y_train_5`) is modified to reflect the specific classification task at hand.

**To the student**: 
* Which image does `some_digit` stand for? (scroll above if needed)
* What is the meaning of the resulting Boolean value (`True` and `False`)?

**Answer**:
* Some_digit represents the image of the digit "4" from the dataset. It is a specific data point that we are using to test our classifier.
* The output False means the classifier predicted that some_digit belongs to the negative class — i.e., the classifier predicted that the image is not a 5.

**To the student**: Regarding the `cross validate` function, browse the sklearn's [docs](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#sklearn.model_selection.cross_validate) for `cross_validate``:
* Describe in your own words what the `cv` argument is for.
* How many times has the model been trained?
* How is the value of `cv` related to the output of this function?
* What do `fit_time`,  `score_time` and `test_score` stand for?

**Answers**
* It specifies the cross-validation splitting strategy. Setting cv=3 partitions the training dataset into 3 equal stratified subsets (folds). In 3 consecutive rounds, 2 folds are combined to train the model, and the remaining fold is used as the evaluation set.

* The model was trained 3 times (once per fold).

* The value of cv determines the length of each array in the returned dictionary. Since cv=3, each array (fit_time, score_time, test_score) contains exactly 3 values, corresponding to the results from each of the 3 iterations.

* What do the three arrays, fit_time, score_time and test_score stand for?
    - `fit_time`: The time (in seconds) taken to train the estimator on the training folds for each CV split.
    - `score_time`: The time (in seconds) taken to compute predictions and evaluate the scoring metric on the validation fold for each split.
    - `test_score`: The resulting performance score (here, classification accuracy) evaluated on each validation fold.


**To the student**: Regrading the accuracy
* Could you provide a textual description of the accuracy result for the first fold? For instance "out of 100 digits, XYZ were correctly predicted, while ZYX were incorrectly predicted"
* Given the uniform distirbution of samples across all 10 digits, and given that the classifier classify one of them versus all the others (`5` versus all others in our case), what would be a simple handwritten rule to achieve 90% accuracy?
* How would you describe the goodness of classification of the `SGDClassifier` versus the expected accuracy of such a handwritten rule? Please describe it in a sentence or two. 


**Answers**
* First-fold accuracy: The result is 0.95035, or 95.035%. Out of every 100 digit images, approximately 95 are correctly classified as “5” or “not 5,” and 5 are incorrectly classified. Exactly 19,007 of the 20,000 images in this fold were classified correctly.

* A simple rule for 90% accuracy: Always predict “not 5.” Assuming all ten digits occur equally often, nine out of ten images are not 5, so this rule achieves 90% accuracy, although it misses every actual 5.

* SGDClassifier versus the rule: The SGDClassifier achieves about 95.7% mean accuracy across the three folds, outperforming the rule’s expected 90% by 5.7 percentage points. However, accuracy alone is insufficient; precision and recall show how effectively it identifies actual 5s.

**To the student**: 
* What is the accuracy of the `DummyClassifier`? 
* How does it compare to the handwritten rule you suggested above, and to the `SGDClassifier`? 

**Answers**

* The DummyClassifier achieves 90.965% accuracy in each of the three folds, as shown by:

In [1]:
'test_score': array([0.90965, 0.90965, 0.90965])

SyntaxError: illegal target for annotation (3812152704.py, line 1)

* It follows the handwritten rule of always predicting “not 5.” Its accuracy is slightly above the expected 90% because the actual training dataset contains 90.965% non-5 digits, rather than exactly 90%.
* The SGDClassifier achieves approximately 95.703% mean accuracy, outperforming the dummy by 4.738 percentage points. Despite its high accuracy, the dummy misses every actual 5, showing why accuracy alone can be misleading.

**To the student**: 
## Exercise - Fashion-MNIST data set

Follow the same process we used to analyze the MNIST data set, but this time do it using the **Fashion-MNIST** data set.

- Get the data set (code cell below)
- Preprocess the data set
- Train a model
- Evaluate the model

> Are you able to replicate the process? Provide your insights.

**Answers** 
Yes. We can repeat the process using Fashion-MNIST, which contains clothing images instead of handwritten digits. Here, label '5' represents a sandal, so our binary task is “sandal or not sandal?”

In [2]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Get the dataset.
fashion_mnist = fetch_openml(
    "Fashion-MNIST", version=1, as_frame=False, parser="auto"
)

# 2. Preprocess: create sandal/not-sandal labels and split the data.
X_fashion = fashion_mnist.data.astype(np.float32)
y_fashion_sandal = fashion_mnist.target.astype(str) == "5"

X_train_fashion = X_fashion[:60000]
X_test_fashion = X_fashion[60000:]
y_train_sandal = y_fashion_sandal[:60000]
y_test_sandal = y_fashion_sandal[60000:]

# Scaling is learned only from training data within each fold.
fashion_clf = make_pipeline(
    StandardScaler(),
    SGDClassifier(random_state=42),
)

# 3. Evaluate using three-fold cross-validation, then train the final model.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

accuracy_scores = cross_val_score(
    fashion_clf,
    X_train_fashion,
    y_train_sandal,
    cv=cv,
    scoring="accuracy",
)
print(f"Mean validation accuracy: {accuracy_scores.mean():.2%}")

fashion_clf.fit(X_train_fashion, y_train_sandal)

# 4. Evaluate on the held-out test set.
y_pred_sandal = fashion_clf.predict(X_test_fashion)

print(f"Test accuracy: {accuracy_score(y_test_sandal, y_pred_sandal):.2%}")
print(
    f"Precision: "
    f"{precision_score(y_test_sandal, y_pred_sandal, zero_division=0):.2%}"
)
print(f"Recall: {recall_score(y_test_sandal, y_pred_sandal):.2%}")
print(f"F1 score: {f1_score(y_test_sandal, y_pred_sandal):.4f}")
print(
    "Confusion matrix:\n",
    confusion_matrix(y_test_sandal, y_pred_sandal, labels=[False, True]),
)

# Compare against an always-majority-class baseline.
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_fashion, y_train_sandal)
print(f"Dummy test accuracy: {dummy.score(X_test_fashion, y_test_sandal):.2%}")

/Users/rangeetha/Applications/VS Editor/Test Project/PerformanceMetricsClassification-Group4/.venv/lib/python3.14/site-packages/sklearn/datasets/_openml.py:131: UserWarning: A network error occurred while downloading https://www.openml.org/api/v1/json/data/list/data_name/fashion-mnist/limit/2/data_version/1. Retrying...
  warn(


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1082)>

To the student: Using `sklearn` resources about Confusion Matrix: [docs](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) and [user guide](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix):

* What are the required arguments that the `confusion_matrix` function expects?
* What is the output of this function, specifically for a binary classifier such as the one tested here?
* What is the meaning of each value in both of the confusion matrics returned above?

**Answer**:
* **Required arguments:** `y_true` (the actual labels) and `y_pred` (the predicted labels). Optional arguments include `labels`, `sample_weight` and `normalize`.
* **Output for a binary classifier:** a 2×2 array where rows are the actual classes and columns are the predicted classes, with the negative class first: `[[TN, FP], [FN, TP]]`.
* **Meaning of the values:**
  * SGDClassifier, `[[53892, 687], [1891, 3530]]`:
    * **TN = 53892**: non-5s correctly predicted as not 5.
    * **FP = 687**: non-5s wrongly predicted as 5.
    * **FN = 1891**: real 5s that the classifier missed (predicted as not 5).
    * **TP = 3530**: real 5s correctly predicted as 5.
    * The values add up to 60,000, the size of the training set.
  * Perfect predictions, `[[54579, 0], [0, 5421]]`:
    * **FP = FN = 0**: there are no errors, so only the diagonal (TN and TP) is filled.
    * The 54,579 non-5s are all TN and the 5,421 real 5s are all TP.
    * This is the best matrix a classifier can get.

**To the student**: 
#### Exercise - Calculate Precision and Recall while evaluating an Autonomous Security Drone

A company has developed an AI-powered security drone to patrol a large warehouse at night. Its primary task is to use its camera to identify if a person is present in a restricted area (the relevant element).

During a simulated test, the drone flies through the warehouse 500 times, analyzing a different camera view each time. The test includes staged scenarios to check the AI's accuracy.

Here is a summary of the test results:

Throughout the 500 views, there were a total of 10 instances where a person was actually present.
The drone's AI system sounded the alarm, claiming it saw a person, 12 times in total.
Upon reviewing the drone's decisions, it was found that of the 12 alarms, 8 of them were correct (a person was truly there). This means 4 alarms were false alarms, triggered by things like shadows or falling boxes.

These are your tasks:

> Calculate the Metrics: Based on the information in the scenario, calculate the Precision and the Recall of the security drone's detection system. Please show the fractions you used for your calculations.

> Explain Your Results (talking points): In your own words, explain what your calculated values for Precision and Recall mean.

**Answer**:

The relevant element is "a person is present".
* TP = 8 (alarms with a person), FP = 12 − 8 = 4 (false alarms), FN = 10 − 8 = 2 (people the drone missed), TN = 500 − 8 − 4 − 2 = 486.
* **Precision** = TP / (TP + FP) = 8 / (8 + 4) = 8 / 12 ≈ **0.667**
* **Recall** = TP / (TP + FN) = 8 / (8 + 2) = 8 / 10 = **0.80**

**Explanation:**
* Precision of 0.667: when the drone sounds the alarm, it is right 2 times out of 3. The other third are false alarms (shadows, falling boxes).
* Recall of 0.80: the drone detects 8 out of the 10 times a person was really there. It misses 2 intruders.
* For a security drone, recall is arguably more important, since a missed intruder is worse than a false alarm. Even so, 2 missed intruders out of 10 is still a gap worth improving.

**To the student**: 
* Similar to the manual calculation of precision, in a new cell below, write down the Python code for calculating recall

In [ ]:
# Values from the confusion matrix of the SGDClassifier: cm = [[53892, 687], [1891, 3530]]
TP = 3530  # cm[1, 1]
FN = 1891  # cm[1, 0]

# Recall = TP / (TP + FN)
recall = TP / (TP + FN)
print(recall)  # 0.6511713705958311, same as recall_score(y_train_5, y_train_pred)

**To the student**: 
* Using the [definition](https://en.wikipedia.org/wiki/F-score#Definition) calculate the F1 score that was generated above, using True Positives (TP), False Negatives (FN) and False Positives (FP). Write this as a Python code in a new cell below.

In [ ]:
# Values from the confusion matrix of the SGDClassifier: cm = [[53892, 687], [1891, 3530]]
TP = 3530  # cm[1, 1] - actual 5s predicted as 5
FN = 1891  # cm[1, 0] - actual 5s predicted as not 5
FP = 687   # cm[0, 1] - non-5s predicted as 5

# F1 = 2TP / (2TP + FP + FN)
f1 = 2 * TP / (2 * TP + FP + FN)
print(f1)  # 0.7325171197343847, same as f1_score(y_train_5, y_train_pred)

# Equivalent: the harmonic mean of precision and recall
precision = TP / (TP + FP)
recall = TP / (TP + FN)
print(2 * precision * recall / (precision + recall))

**Answer**:
F1 = 2TP / (2TP + FP + FN) = 2 × 3530 / (2 × 3530 + 687 + 1891) = 7060 / 9638 ≈ **0.7325**, which matches the value from `f1_score` above. It is the harmonic mean of precision (0.837) and recall (0.651), so it sits closer to the lower of the two than the arithmetic mean (0.744) would.

**To the student**: 
* Assume that you are developing an ML classifeir for medical diagnostics, with the propose of being a first indicator to identify whether a patient has a medical condition that requires treatment or not. Based on this result, the data wil be delivered to a medical professional or not (this might not be a great practice, but let's assume it for the sake of the example). What would be better, a high precision or a high recall? 
* Likewise, what would be better to optimize for (a high precision versus recall), when an ML classifier should highlight the best investment opportunities in a stock exchange?
* Add two more real-world cases in which:
  * Would a high recall will be desired, even on the expense of a lower precision? 
  * And when would a high precision will be preferred, on the expense of a lower recall? 


**Answer**:
* **Medical diagnostics (first screening): high recall.** Missing a sick patient (a false negative) means they never reach a doctor and go untreated, which can be dangerous. A false positive only means a medical professional reviews a healthy patient's data and rules the condition out. That is extra work, but the professional acts as a safety net that filters out false alarms. So we accept lower precision to catch as many real cases as possible.
* **Stock-exchange investment opportunities: high precision.** Every stock the model flags is money we may put at risk, so a false positive (a "great opportunity" that turns out bad) costs real money. Missing some good opportunities (false negatives) is acceptable, because there are always other investments to choose from. We want the few stocks the model flags to be very reliable.
* **High recall preferred, even at lower precision:**
  1. **Credit-card fraud detection:** missing a fraudulent transaction costs the bank and customer money. Flagging a legitimate one only costs a quick verification text or call.
  2. **Airport security screening (weapons in luggage):** letting a weapon through is unacceptable. A false alarm only means a manual bag check.
* **High precision preferred, even at lower recall:**
  1. **Spam filtering:** sending an important legitimate email (job offer, bill) to the spam folder is worse than letting a few spam emails reach the inbox.
  2. **Kid-safe video recommendations:** every video shown to children must really be safe. It is fine if many safe videos are rejected (low recall), as long as no inappropriate video gets through.

**To the student**: 
* How do precision and recall change with increasing or decreasing the threshold? 
* What is the intuition behind this behavior?

**Answer**:
* **Increasing the threshold → precision goes up, recall goes down.** **Decreasing the threshold → recall goes up, precision goes down.** The notebook shows this. At threshold 0, precision ≈ 0.84 and recall ≈ 0.65. Raising the threshold to ≈ 3,370 brings precision to 0.90, but recall drops to ≈ 0.48.
* **Intuition:** the threshold is how sure the classifier must be (how high the decision score must be) before it says "5".
  * A **higher threshold** makes the classifier stricter. It only says "5" when very confident, so fewer non-5s slip through (fewer FP → higher precision). But more real 5s with lower scores are now rejected (more FN → lower recall).
  * A **lower threshold** makes it more permissive. It catches almost every real 5 (higher recall), but also labels more non-5s as 5 (more FP → lower precision).
* **Recall changes smoothly, precision can be bumpy.** The denominator of recall, TP + FN, is the fixed number of actual 5s. So recall can only stay the same or decrease as the threshold rises. The denominator of precision, TP + FP, shrinks as the threshold rises, so precision can occasionally dip. For example, if raising the threshold removes a true positive but no false positives, precision goes down. That is why the precision curve is jagged at high thresholds while the recall curve is smooth.
* Precision and recall therefore trade off against each other, and the threshold is chosen based on which error (FP or FN) is more costly for the application.